# 01 · One Brent application, by hand

Goal: fetch **one** planning application for Brent, with a handful of flat fields only
(no nested lists), and understand every part of the request and the response.

You write the code. Each step has:
- **PDF:** what the API document says (the GLA *API connection technical documentation*);
- **httpx:** the bit of httpx you need;
- an empty cell with hints in comments.

> **Golden rule:** Brent and Lewisham only. The PDF's examples use other boroughs
> (`Newham-701491`, `lpa_name.raw = Lambeth`). **Do not run them** — copy the *shape*,
> change the borough to Brent.

## Step 1 · Where the API lives

**PDF, "Steps to connect", point 1a:** the guest endpoint is
`https://planningdata.london.gov.uk/api-guest/`. Point 1b: no authorisation.

**PDF, "Examples":** every URL is `<base>/<index>/<action>`, e.g.
`.../api-guest/applications/_search`. "Available Indices" lists the indexes;
`applications` is the one with everything in it.

Write two variables: the base URL (no trailing slash) and the index name.

In [5]:
import httpx
import json
from pathlib import Path
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

HEADERS = {
    "X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"],
    "Content-Type": "application/json",
}

BASE_URL = 'https://planningdata.london.gov.uk/api-guest'
response = httpx.get(f"{BASE_URL}/_all", headers=HEADERS, timeout=30)
indexes = response.json()
list(indexes)

['applications',
 'existing_proposed_floorspace_details',
 'non_permanent_dwellings_details',
 'open_spaces_details',
 'other_resi_accommodation_unit_details',
 'protected_spaces_details',
 'residential_units']

## Step 3 · httpx in five lines

httpx is a requests-style HTTP client. What you need:

| You want | httpx |
|---|---|
| send a POST with a JSON body | `httpx.post(url, headers=..., json=body, timeout=30)` |
| send a GET | `httpx.get(url, headers=..., timeout=30)` |
| the HTTP status | `response.status_code` (200 = OK) |
| stop on errors | `response.raise_for_status()` (raises for 4xx/5xx) |
| the body as Python dicts/lists | `response.json()` |
| the raw text (useful when it isn't JSON) | `response.text` |
| how long it took | `response.elapsed` |

Notes:
- `json=body` turns a Python dict into JSON **and** sets `Content-Type: application/json`
  for you. (`data=` would send form data — not what Elasticsearch wants.)
- Always pass `timeout=`; the default is 5 s.
- The cURL flags in the PDF map one-to-one: `--request POST` → `httpx.post`,
  `--header` → `headers=`, `--data-raw` → `json=`.

## Step 4 · Which action? `_source/{id}` vs `_search`

The PDF shows two:

1. **"Retrieve data for a specific application"** → `GET <base>/applications/_source/<id>`.
   Needs an id you already know, and returns the **whole** document, nested lists and
   all. Not what we want yet.
2. **"Search for local planning authority ... returning specific data fields"** →
   `POST <base>/applications/_search` with a JSON body. Lets us filter by borough and
   pick which fields come back. **Use this one.**

(It's a POST, but it only reads. Elasticsearch uses POST because GET bodies are
unreliable.)

Write the URL for the search action from `BASE_URL` and `INDEX`.

In [19]:
BASE_URL = 'https://planningdata.london.gov.uk/api-guest'
body = {
    "size": 1,                                                    # how many records
    "query": {"bool": {"filter": [{"term": {"borough.raw": "Brent"}}]}},
}
#response = httpx.get(f"{BASE_URL}/applications/_source/Brent-22_4070", headers=HEADERS, json=body, timeout=30)
response = httpx.get(f"{BASE_URL}/applications/_source/Brent-22_4070", headers=HEADERS, timeout=30)
data = response.json()
data
#data["hits"]["hits"][0]["_source"]

{'reference_no_of_permission_being_relied_on': None,
 'title_number': None,
 'application_type_full': 'Prior Approval: Development for electronic communications network',
 'lpa_app_no': '22/4070',
 'last_synced': '2025-04-07T09:29:29.787Z',
 'url_planning_app': 'https://pa.brent.gov.uk/online-applications/applicationDetails.do?keyVal=DCAPR_162863&activeTab=summary',
 'actual_completion_date': None,
 'pp_id': 'PP-11729739',
 'lapsed_date': '23/01/2026',
 'secondary_street_name': None,
 'centroid': {'lon': -0.225784, 'lat': 51.533768},
 'decision_process': 'Delegated',
 'actual_commencement_date': None,
 'site_number': None,
 'uprn': '202113435',
 'epc_number': None,
 'appeal_status': None,
 'last_date_consultation_comments': '04/01/2023',
 'description': 'Prior approval for proposed telecommunications installation comprising a 20-metre-high monopole supporting 6 no. antennas and 2 no. transmission dishes, 2 no. equipment cabinets and development works ancillary thereto on footpath adjac

## Step 5 · The body: one Brent row, flat fields only

Start from the PDF's search body and change it:

| PDF example | What we change | Why |
|---|---|---|
| `"term": {"lpa_name.raw": "Lambeth"}` | `"term": {"borough.raw": "Brent"}` | golden rule; `borough` = where the site is. `.raw` = exact match on the whole value |
| `"term": {"application_type.raw": ...}` | drop it | not needed yet |
| `"range": {"valid_date": ...}` | drop it for now | not needed yet (note the PDF has a stray space in `" 01/01/2021"`) |
| `"_source": [...]` | keep the idea | it chooses which fields come back; list only flat ones |
| *(not in the PDF)* `"size": 1` | add | how many hits to return; Elasticsearch default is 10. From the Elasticsearch 7.9 search docs linked in the PDF |

Body shape (fill in the `...`):

```python
body = {
    "size": ...,
    "query": {"bool": {"filter": [ {"term": {...: ...}} ]}},
    "_source": [ ... ],
}
```

Suggested first fields (all flat): `id`, `lpa_app_no`, `lpa_name`, `borough`, `ward`,
`site_name`, `description`, `application_type`, `valid_date`, `decision`,
`decision_date`, `status`. The PDF's "Fields" list points to the technical schema
(the xlsx) for the rest.

`filter` vs the PDF's `must`: both mean "must match". `filter` skips relevance
scoring — we don't need a score to pick one row.

In [ ]:
# body = {
#     ...
# }

## Step 6 · Send it

Use Step 3's table: post the body to `search_url` with your headers and a timeout,
then check the status code before anything else.

If it isn't 200, print `response.text` — Elasticsearch explains its errors there
(a typo in a field name, bad JSON, ...).

In [ ]:
# response = httpx.post(...)
# response.status_code

## Step 7 · Read the response envelope

`response.json()` is a dict. Look at its **top-level keys** first, before the data:

- `took`: milliseconds Elasticsearch spent;
- `timed_out`;
- `_shards`: internal, ignore;
- `hits`:
  - `hits["total"]`: how many records matched (note: capped at 10,000 unless you ask
    for an exact count — that's `"track_total_hits": true` in the body, a later step);
  - `hits["hits"]`: the list of results, `size` long.

Each item in `hits["hits"]` has `_index`, `_id`, `_score` and **`_source`** — the
application itself, with only the fields you asked for.

Print the keys at each level, then the one `_source`.

In [ ]:
# data = response.json()
# data.keys()

In [ ]:
# hit = ...
# hit.keys()

In [ ]:
# hit["_source"]

## Step 8 · Questions to answer yourself

Write the answers in a markdown cell below.

1. Does `id` match `lpa_name` + `-` + `lpa_app_no`? What happened to the `/`?
   (PDF: "Certain characters in a reference will be replaced with underscores".)
2. What format are the dates in?
3. Which of your fields came back missing or empty? Missing key vs `null` vs `""` —
   are they different here?
4. Run it again. Is it the same row? (No `sort` → Elasticsearch's order is not
   guaranteed.) What would you add to always get, say, the newest application?
5. Add one field that doesn't exist (e.g. `"not_a_field"`) to `_source`. Error, or
   silently missing? What does that mean for typos?

## Your notes